# Dependencies

In [2]:
import torch
import subprocess
import time

In [ ]:
WORLD_SIZE = 2

In [ ]:

# configs
torch.cuda.empty_cache()
torch.cuda.synchronize()
NUM_CONTAINERS = WORLD_SIZE
IMAGE_NAME = "moe_expert"
BASE_PORT = 5000
layer = 1
experts_count_per_container = 8//NUM_CONTAINERS

docker_start_time = time.time() 

for i in range(NUM_CONTAINERS):
    cmd = [
    "docker", "run", "-d",
    "--name", f"layer{layer}_exp_{i*experts_count_per_container}_{(i+1)*experts_count_per_container-1}",
    "--gpus", "all",
    "--rm",
    "-p", f"{BASE_PORT+i}:5000",
    
    "-v", "/root/MG_test/mixtral/mixtral-mcore-TP1PP1EP2Layer1:/app/weights",
    "-v", "/root/MG_test/mixtral/REPLICATE/saved_objects:/app/saved_objects",
    "-e", f"RANK={i}",
    "-e", f"NUM_LOCAL_EXPERTS={experts_count_per_container}",
    "-e", f"GPU_IDX={0}",
    "-e", f"WEIGHT_PATH=/app/weights",
    "-e", f"PATH_SAVEDOBJ=/app/saved_objects",
    "-e", f"LAYER={layer}",
    "-e", f"WARMUP={True}",
    
    
    IMAGE_NAME
    ]
    try:
        subprocess.run(cmd, check=True)
        print(f"fused_moe_layer_{layer}_exp_{i*experts_count_per_container}_{(i+1)*experts_count_per_container-1}\n容器启动成功！")
    except subprocess.CalledProcessError as e:
        print(f"启动失败: {e}")
        
    # duplicated container    
    cmd = [
    "docker", "run", "-d",
    "--name", f"layer{layer}_exp_{(i+NUM_CONTAINERS)*experts_count_per_container}_{(i+NUM_CONTAINERS+1)*experts_count_per_container-1}",
    "--gpus", "all",
    "--rm",
    "-p", f"{BASE_PORT+i+NUM_CONTAINERS}:5000",
    
    "-v", "/root/MG_test/mixtral/mixtral-mcore-TP1PP1EP2Layer1:/app/weights",
    "-v", "/root/MG_test/mixtral/REPLICATE/saved_objects:/app/saved_objects",
    "-e", f"RANK={i}",
    "-e", f"NUM_LOCAL_EXPERTS={experts_count_per_container}",
    "-e", f"GPU_IDX={0}",
    "-e", f"WEIGHT_PATH=/app/weights",
    "-e", f"PATH_SAVEDOBJ=/app/saved_objects",
    "-e", f"LAYER={layer}",
    "-e", f"WARMUP={True}",
    # "-e", f"LOADED_EXPERTS={list(range(i*experts_count_per_container,(i+1)*experts_count_per_container))}",

    
    
    IMAGE_NAME
    ]
    try:
        subprocess.run(cmd, check=True)
        print(f"fused_moe_layer_{layer}_exp_{(i+NUM_CONTAINERS)*experts_count_per_container}_{(i+NUM_CONTAINERS+1)*experts_count_per_container-1}\n容器启动成功！")
    except subprocess.CalledProcessError as e:
        print(f"启动失败: {e}")
        

end_time = time.time()  # 记录结束时间
elapsed_time = end_time - docker_start_time  # 计算用时
print(f"Containers launched in: {elapsed_time*1000:.2f} ms")
print(f"Average containers launching time : {elapsed_time*1000/WORLD_SIZE:.2f} ms")

In [ ]:
import pickle
RANK=0
with open(f"/root/MG_test/mixtral/REPLICATE/saved_objects/rank_{RANK}/args.pickle", 'rb') as f:
    args = pickle.load(f)



Namespace(num_layers=1, encoder_num_layers=1, decoder_num_layers=None, hidden_size=4096, ffn_hidden_size=14336, num_attention_heads=32, attention_backend=<AttnBackend.auto: 5>, kv_channels=128, group_query_attention=True, num_query_groups=8, max_position_embeddings=32768, position_embedding_type='rope', relative_attention_num_buckets=32, relative_attention_max_distance=128, use_rotary_position_embeddings=False, rotary_base=1000000, rotary_percent=1.0, rotary_interleaved=False, rotary_seq_len_interpolation_factor=None, use_rope_scaling=False, rope_scaling_factor=8.0, add_position_embedding=False, mrope_section=None, make_vocab_size_divisible_by=128, normalization='RMSNorm', norm_epsilon=1e-05, apply_layernorm_1p=False, apply_residual_connection_post_layernorm=False, openai_gelu=False, squared_relu=False, swiglu=True, onnx_safe=None, bert_binary_head=True, untie_embeddings_and_output_weights=True, multi_latent_attention=False, mtp_num_layers=None, mtp_loss_scaling_factor=0.1, attention_d

In [3]:
print(type(args))

<class 'argparse.Namespace'>


In [ ]:
import torch 
a=torch.load("/home/ubuntu/MG_test/mixtral/mixtral-mcore-TP1PP1EP1Layer0_1/iter_0000001/mp_rank_00/model_optim_rng.pt",weights_only=False)

In [ ]:
k=a["model"]
fc1= torch.cat([
                k["decoder.layers.0.mlp.experts.linear_fc1.weight0"],
                k["decoder.layers.0.mlp.experts.linear_fc1.weight0"]
            ], dim=0)
print(k["decoder.layers.0.mlp.experts.linear_fc1.weight0"].shape)
print(k["decoder.layers.0.mlp.experts.linear_fc2.weight0"].shape)


RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 4096 but got size 14336 for tensor number 1 in the list.

In [3]:
from safetensors.torch import load_file
import torch

file_path1 = '/home/ubuntu/MG_test/checkpoints/model-00001-of-00019.safetensors' # replace with your actual file path
file_path2 = '/home/ubuntu/MG_test/checkpoints/model-00002-of-00019.safetensors' # replace with your actual file path
loaded = {}
loaded.update(load_file(file_path1))
loaded.update(load_file(file_path2))
loaded.keys()


dict_keys(['model.embed_tokens.weight', 'model.layers.0.block_sparse_moe.experts.0.w1.weight', 'model.layers.0.block_sparse_moe.experts.0.w2.weight', 'model.layers.0.block_sparse_moe.experts.0.w3.weight', 'model.layers.0.block_sparse_moe.experts.1.w1.weight', 'model.layers.0.block_sparse_moe.experts.1.w2.weight', 'model.layers.0.block_sparse_moe.experts.1.w3.weight', 'model.layers.0.block_sparse_moe.experts.2.w1.weight', 'model.layers.0.block_sparse_moe.experts.2.w2.weight', 'model.layers.0.block_sparse_moe.experts.2.w3.weight', 'model.layers.0.block_sparse_moe.experts.3.w1.weight', 'model.layers.0.block_sparse_moe.experts.3.w2.weight', 'model.layers.0.block_sparse_moe.experts.3.w3.weight', 'model.layers.0.block_sparse_moe.experts.4.w1.weight', 'model.layers.0.block_sparse_moe.experts.4.w2.weight', 'model.layers.0.block_sparse_moe.experts.4.w3.weight', 'model.layers.0.block_sparse_moe.experts.5.w1.weight', 'model.layers.0.block_sparse_moe.experts.5.w2.weight', 'model.layers.0.block_spa

In [2]:
import pickle
with open("/home/ubuntu/MG_test/mixtral/REPLICATE/saved_objects/rank_0/config.pickle", 'rb') as f:
    config = pickle.load(f)
print(config)

TransformerConfig(tensor_model_parallel_size=1, pipeline_model_parallel_comm_backend=None, pipeline_model_parallel_size=1, virtual_pipeline_model_parallel_size=None, sequence_parallel=False, context_parallel_size=1, hierarchical_context_parallel_sizes=None, expert_model_parallel_size=2, expert_tensor_parallel_size=1, moe_extended_tp=False, perform_initialization=True, use_cpu_initialization=None, fp16=False, bf16=True, params_dtype=torch.bfloat16, timers=None, finalize_model_grads_func=None, grad_scale_func=None, no_sync_func=None, grad_sync_func=None, param_sync_func=None, deterministic_mode=False, enable_autocast=False, autocast_dtype=torch.bfloat16, num_microbatches_with_partial_activation_checkpoints=None, gradient_accumulation_fusion=False, async_tensor_model_parallel_allreduce=True, use_te_rng_tracker=False, tp_comm_overlap=False, tp_comm_bulk_wgrad=True, tp_comm_bulk_dgrad=True, tp_comm_overlap_ag=True, tp_comm_overlap_rs=True, tp_comm_overlap_rs_dgrad=False, tp_comm_split_ag=Tr

In [4]:
import os
save_weights_dir="/home/ubuntu/MG_test/weights/"
for layer in range(2):
    for expert in range(8):
        w1_prefix = f'model.layers.{layer}.block_sparse_moe.experts.{expert}.w1.weight'
        w2_prefix = f'model.layers.{layer}.block_sparse_moe.experts.{expert}.w2.weight'
        w3_prefix = f'model.layers.{layer}.block_sparse_moe.experts.{expert}.w3.weight'
        target_prefix_fc1 = f"decoder.layers.{layer}.mlp.experts.linear_fc1.weight{expert}"
        target_prefix_fc2 = f"decoder.layers.{layer}.mlp.experts.linear_fc2.weight{expert}"
        target_prefix_fc1=save_weights_dir+target_prefix_fc1.replace(".","_")+".pt"
        target_prefix_fc2=save_weights_dir+target_prefix_fc2.replace(".","_")+".pt"
        fc1= torch.cat([
                loaded[w1_prefix],
                loaded[w3_prefix],
            ], dim=0)
        
        fc2= loaded[w2_prefix]
        
        torch.save(fc1,target_prefix_fc1)
        torch.save(fc2,target_prefix_fc2)
        

In [2]:
import os
import torch
save_weights_dir="/root/MG_test/weights/"
target_prefix_fc1 = f"decoder.layers.0.mlp.experts.linear_fc1.weight0"
path=target_prefix_fc1.replace(".","_")+".pt"
weight_fc1 = torch.load(path,weights_only=True)
weight_fc1

tensor([[-3.3569e-03,  1.0437e-02, -1.5747e-02,  ..., -1.6235e-02,
          5.2261e-04, -1.3062e-02],
        [-1.2360e-03,  2.0752e-03, -1.6968e-02,  ...,  1.7944e-02,
          1.0620e-02, -3.4943e-03],
        [-1.5869e-02,  1.9897e-02,  1.8845e-03,  ...,  1.8433e-02,
         -1.8921e-03, -5.2795e-03],
        ...,
        [-4.0894e-03,  1.5564e-02,  4.2152e-04,  ...,  5.8899e-03,
         -6.5994e-04,  7.8735e-03],
        [ 1.7334e-02,  7.5073e-03, -1.2146e-02,  ...,  1.6968e-02,
          4.5776e-03,  1.7456e-02],
        [ 2.1240e-02,  1.2756e-02,  1.0559e-02,  ...,  5.8289e-03,
          7.2937e-03,  6.6280e-05]], dtype=torch.bfloat16)

# Send request

In [5]:
import torch
import requests
tokens_per_expert = torch.tensor([32]*4, dtype=torch.int32).cpu().tolist()
dispatched_input = torch.randn(1, 32*4, 4096,
                                   dtype=torch.bfloat16).cpu().tolist()

url="http://localhost:8000/forward"

response = requests.post(
    url,
    json={
        'dispatched_input':dispatched_input,
        'tokens_per_expert':tokens_per_expert,
        'layer':'0'
        }  # 自动将字典转换为 JSON
)
hidden=torch.tensor(response.json()["hidden_output"]).cuda()

print(hidden.shape)


torch.Size([1, 128, 4096])


# Build Container

In [15]:
! sudo docker build -t expert_container .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

Sending build context to Docker daemon  5.328MB
Step 1/7 : FROM nvidia/cuda:12.6.2-cudnn-devel-ubuntu24.04
 ---> 7cb2db509a78
Step 2/7 : WORKDIR /app
 ---> Using cache
 ---> b8c2f7287896
Step 3/7 : RUN apt-get update && apt-get install -y git python3 python3-pip python3-dev python3.12-venv && rm -rf /var/lib/apt/lists/*
 ---> Running in bcfba6e2ea6e


Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1581 B]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [22.1 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1129 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1496 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1108 kB]
Get:13 http://archive.

In [1]:
import subprocess


NUM_CONTAINERS = 2
IMAGE_NAME = "expert_container"
BASE_PORT = 5000
layer = [0]
experts_count_per_container = 60//NUM_CONTAINERS


experts=[[0,1,2,3],[4,5,6,7]]
for i in range(NUM_CONTAINERS):
    cmd = [
    "docker", "run", "-d",
    "--name", f"moe_layer_{layer}_exp_{experts[i][0]}_{experts[i][-1]}",
    "--gpus", "all",
    "--rm",
    "-p", f"{BASE_PORT+i}:5000",
    "-v", "/home/ubuntu/MG_test/weights:/app/weights",
    "-v", "/home/ubuntu/MG_test/mixtral/REPLICATE/saved_objects:/app/saved_objects",
    "-e", f"RANK={i}",
    "-e", f"EXPERTS={[experts[i]]}",
    "-e", f"GPU_IDX={0}",
    "-e", f"WEIGHT_PATH=/app/weights",
    "-e", f"LAYER={layer}",
    "-e", f"PATH_SAVEDOBJ=/app/saved_objects",
    
    IMAGE_NAME
    ]
    try:
        # print(' '.join(cmd))
        subprocess.run(cmd, check=True)
        print(f"moe_layer_exp_{experts[i][0]}_{experts[i][-1]}\n容器启动成功！")
    except subprocess.CalledProcessError as e:
        print(f"启动失败: {e}")

启动失败: Command '['docker', 'run', '-d', '--name', 'moe_layer_[0]_exp_0_3', '--gpus', 'all', '--rm', '-p', '5000:5000', '-v', '/home/ubuntu/MG_test/weights:/app/weights', '-v', '/home/ubuntu/MG_test/mixtral/REPLICATE/saved_objects:/app/saved_objects', '-e', 'RANK=0', '-e', 'EXPERTS=[[0, 1, 2, 3]]', '-e', 'GPU_IDX=0', '-e', 'WEIGHT_PATH=/app/weights', '-e', 'LAYER=[0]', '-e', 'PATH_SAVEDOBJ=/app/saved_objects', 'expert_container']' returned non-zero exit status 125.
启动失败: Command '['docker', 'run', '-d', '--name', 'moe_layer_[0]_exp_4_7', '--gpus', 'all', '--rm', '-p', '5001:5000', '-v', '/home/ubuntu/MG_test/weights:/app/weights', '-v', '/home/ubuntu/MG_test/mixtral/REPLICATE/saved_objects:/app/saved_objects', '-e', 'RANK=1', '-e', 'EXPERTS=[[4, 5, 6, 7]]', '-e', 'GPU_IDX=0', '-e', 'WEIGHT_PATH=/app/weights', '-e', 'LAYER=[0]', '-e', 'PATH_SAVEDOBJ=/app/saved_objects', 'expert_container']' returned non-zero exit status 125.


docker: Error response from daemon: Invalid container name (moe_layer_[0]_exp_0_3), only [a-zA-Z0-9][a-zA-Z0-9_.-] are allowed.
See 'docker run --help'.
docker: Error response from daemon: Invalid container name (moe_layer_[0]_exp_4_7), only [a-zA-Z0-9][a-zA-Z0-9_.-] are allowed.
See 'docker run --help'.
